# 🧪 Track A Challenge Evaluator

This notebook provides the following evaluations:

- **Validation:** Segmentation-distance evaluation between simulated and translated images.
- **Realism:** Distribution-level and single-image metrics comparing real images with translated images.
- **Behavioural realism:** Agreement of ISU predictions for simulated and translated images, evaluated with the selected failure-detection model.
- **Failure detection and diversity:** Diversity of failures detected by ISU when processing the images, evaluated with the selected failure-detection model.
- **Efficiency:** Translation time measured on specified hardware.

The evaluator writes raw and per-contestant reports to `track_a_evaluation_output`. Cross-contestant Pareto ranking is handled separately in the organizer-only ranking notebook.

In [1]:
import gc
import importlib
import json
import sys
from pathlib import Path

# The evaluator may be opened from the repository root or from track_a/.
workspace_dir = Path.cwd()
venv_candidates = (
    workspace_dir / 'venv' / 'bin' / 'python',
    workspace_dir.parent / 'venv' / 'bin' / 'python',
)
VENV_PYTHON = next((path.resolve() for path in venv_candidates if path.exists()), None)
ACTIVE_PYTHON = Path(sys.executable).resolve()
if VENV_PYTHON is None or ACTIVE_PYTHON != VENV_PYTHON:
    expected = str(VENV_PYTHON) if VENV_PYTHON else 'the repository venv/bin/python'
    raise RuntimeError(
        f'Select the Track A venv kernel and restart the notebook. '
        f'Expected: {expected}; active: {ACTIVE_PYTHON}'
    )

import track_a_metrics
importlib.reload(track_a_metrics)
from image_metrics_utils import (
    MoondreamAnalyzer,
    Qwen25VLAnalyzer,
    TransformersSamSegmenter,
    save_report,
)
from track_a_metrics import *

OUTPUT = Path('track_a_evaluation_output')
OUTPUT.mkdir(exist_ok=True)
contestants = discover_contestants()
reports = {name: {'contestant': name} for name in contestants}
print(f'Found {len(contestants)} contestant(s): {list(contestants)}')

Found 3 contestant(s): ['your_team_name', 'your_team_name copy', 'your_team_name copy 2']


## 🧩 SAM segmentation distance

This cell measures the structural distance between each simulated source image and its generated translation. A class-agnostic Segment Anything Model (SAM) creates object partitions for both images, and the evaluator compares those partitions without using semantic class names.

The comparison is paired by image stem:

- **Simulated vs. generated:** checks whether the translation preserves the source scene's object boundaries and spatial structure.
- Real reference images are **not** used in this SAM calculation because they are an unpaired reference distribution. Their visual similarity is evaluated in the later realism section.

The report contains partition-agreement metrics including boundary F1, symmetric region covering, matched IoU, adjusted Rand index (ARI), and normalized mutual information (NMI). These metrics are reported together with `semantic_segmentation_score = 1 - matched_iou`, which acts as the segmentation distance: lower values indicate closer structural agreement, while the agreement metrics are generally better when higher. The result is a continuous diagnostic; no pass/fail threshold is applied.

In [2]:
sam = TransformersSamSegmenter('facebook/sam-vit-base')
for index, (name, folders) in enumerate(contestants.items(), 1):
    print(f'Contestant {index}/{len(contestants)}: SAM distance for {name}')
    reports[name]['sam'] = evaluate_sam(folders, OUTPUT / name / 'sam', sam)
    save_report(reports[name], OUTPUT / f'{name}.json')
del sam
gc.collect()
print('SAM distance complete.')

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
Device set to use cuda:0


Contestant 1/3: SAM distance for your_team_name
mapping: {'simulated': {'sample_0001': PosixPath('submissions/your_team_name/simulated/sample_0001_sim.png')}, 'generated': {'sample_0001': PosixPath('submissions/your_team_name/generated/sample_0001.png')}}
common stems: {'sample_0001'}
Found 1 pairs for segmentation evaluation.
Pair details: {'stem': 'sample_0001', 'simulated': PosixPath('submissions/your_team_name/simulated/sample_0001_sim.png'), 'generated': PosixPath('submissions/your_team_name/generated/sample_0001.png')}
Processing pair 1/1: sample_0001
  SAM simulated vs generated: 1/1 - sample_0001 (cached)
Contestant 2/3: SAM distance for your_team_name copy
mapping: {'simulated': {'sample_0001': PosixPath('submissions/your_team_name copy/simulated/sample_0001_sim.png')}, 'generated': {'sample_0001': PosixPath('submissions/your_team_name copy/generated/sample_0001.png')}}
common stems: {'sample_0001'}
Found 1 pairs for segmentation evaluation.
Pair details: {'stem': 'sample_0001

## 1. 📊 Realism evaluation

This section evaluates visual similarity and distribution-level realism using public image metrics. The evaluator reports results separately for generated-versus-real and simulated-versus-real comparisons, so the generated images can be compared with the original baseline.

It applies two groups of metrics:

- **Paired image metrics:** exact pixel match, correlation coefficient, histogram intersection, LBP-histogram similarity, PSNR, SSIM, normalized mutual information, fractal-dimension similarity, KL divergence, MSE, VGG perceptual distance, GLCM texture distance, and Wasserstein distance. These compare corresponding image pairs and assess pixel, structure, texture, histogram, and perceptual differences.
- **Distribution metrics:** Fréchet Inception Distance (FID), Kernel Inception Distance (KID), and Inception Score. These compare image distributions rather than individual pixels: lower FID and KID indicate closer distributions, while higher Inception Score is preferred.

For paired metrics, similarity measures such as SSIM and PSNR are generally better when higher, while distance or error measures such as MSE, KL divergence, Wasserstein distance, and VGG perceptual distance are better when lower. The raw values are reported for analysis; the later ranking step determines how they contribute to comparisons across participants.

In [3]:
for index, (name, folders) in enumerate(contestants.items(), 1):
    print(f'Contestant {index}/{len(contestants)}: public realism for all scenes in {name}')
    reports[name]['realism_raw'] = evaluate_public_realism(folders)
    save_report(reports[name], OUTPUT / f'{name}.json')
print('Public realism metrics complete.')


Contestant 1/3: public realism for all scenes in your_team_name
mapping: {'generated': {'sample_0001': PosixPath('submissions/your_team_name/generated/sample_0001.png')}, 'real': {'sample_0001': PosixPath('submissions/your_team_name/reference/sample_0001.jpg')}}
common stems: {'sample_0001'}
Found 1 pairs for generated vs real evaluation.
  Loading VGG16...
  generated vs real: 1/1 - sample_0001
mapping: {'simulated': {'sample_0001': PosixPath('submissions/your_team_name/simulated/sample_0001_sim.png')}, 'real': {'sample_0001': PosixPath('submissions/your_team_name/reference/sample_0001.jpg')}}
common stems: {'sample_0001'}
Found 1 pairs for simulated vs real evaluation.
  Loading VGG16...
  simulated vs real: 1/1 - sample_0001
Contestant 2/3: public realism for all scenes in your_team_name copy
mapping: {'generated': {'sample_0001': PosixPath('submissions/your_team_name copy/generated/sample_0001.png')}, 'real': {'sample_0001': PosixPath('submissions/your_team_name copy/reference/samp

## 2. 🔍 Failure detection

This section queries categorical in-car scene features with a selectable deterministic VLM baseline. Set `FAILURE_MODEL` in the code cell below to choose between:

- **`moondream`** — the lightweight option. It requires fewer resources and is intended to run on CPU, making it suitable for local smoke tests and machines without a GPU.
- **`qwen`** — Qwen2.5-VL-3B-Instruct. It generally requires more memory and compute, but provides a stronger multimodal baseline when suitable hardware is available.

For each evaluated stem, the selected model runs on the generated image and its matched real reference image, then compares the predicted feature values. The label JSON files determine which public feature keys are evaluated. A scene is counted as a failure when at least one evaluated feature differs between the generated and real-image predictions.

The report records per-feature matches, raw model answers, inference time, the selected model configuration, the number of failures, the number of executed scenes, and the overall failure rate:

`failure_rate = failures / executed`

This is a behavioural comparison rather than a direct image-quality metric. The public model is a baseline; the designated industrial SUT is used for final evaluation when required.

In [4]:
# Choose 'moondream' for the lightweight CPU-friendly baseline or
# 'qwen' for the stronger Qwen2.5-VL-3B-Instruct baseline.
FAILURE_MODEL = 'moondream'

import image_metrics_utils
importlib.reload(image_metrics_utils)
importlib.reload(track_a_metrics)

evaluate_failure_rate = track_a_metrics.evaluate_failure_rate
if FAILURE_MODEL == 'moondream':
    analyzer = MoondreamAnalyzer(revision='2025-06-21', device='cpu', seed=0)
elif FAILURE_MODEL == 'qwen':
    analyzer = Qwen25VLAnalyzer(model_name='Qwen/Qwen2.5-VL-3B-Instruct')
else:
    raise ValueError("FAILURE_MODEL must be either 'moondream' or 'qwen'.")

print(f'Using failure-detection model: {FAILURE_MODEL}')
for index, (name, folders) in enumerate(contestants.items(), 1):
    print(f'Contestant {index}/{len(contestants)}: {FAILURE_MODEL} failure evaluation for {name}')
    failure_report = evaluate_failure_rate(folders, analyzer)
    reports[name]['failure_raw'] = failure_report
    failure_rate = failure_report['failure_rate']
    failure_text = f'{failure_rate:.3f}' if failure_rate is not None else 'N/A'
    print(
        f'  Failure rate: {failure_text} '
        f"({failure_report['failures']}/{failure_report['executed']})"
    )
    save_report(reports[name], OUTPUT / f'{name}.json')
del analyzer
gc.collect()
print(f'{FAILURE_MODEL} failure-rate evaluation complete.')

Using failure-detection model: moondream
Contestant 1/3: moondream failure evaluation for your_team_name
  Failure rate: N/A (0/0)
Contestant 2/3: moondream failure evaluation for your_team_name copy
  Failure rate: N/A (0/0)
Contestant 3/3: moondream failure evaluation for your_team_name copy 2
  Failure rate: N/A (0/0)
moondream failure-rate evaluation complete.


## 3. 📋 Final report and failure diversity

This section assembles the previously computed SAM, image-realism, and failure-detection results into one final report for each contestant. It does not remove scenes or filter the raw metric sections.

It then measures diversity among the scenes flagged as failures by the selected failure-detection model (`FAILURE_MODEL`):

- **Feature diversity:** computes the mean pairwise Hamming distance across the categorical label values of failing scenes. A value of `0` means the failing scenes have identical evaluated feature values; larger values indicate more varied failure cases. The report also records the number of failing scenes and the number of unique values per feature.
- **CLIP visual diversity:** computes the mean pairwise cosine distance between CLIP embeddings of the generated images for failing scenes. This is a visual-diversity measure and is `0` when fewer than two failing scenes are available.

The final report is written to `track_a_evaluation_output/<contestant>.json` and retains the raw per-image results, failure details, realism metrics, and diversity summaries.

In [5]:
for index, (name, folders) in enumerate(contestants.items(), 1):
    print(f'Contestant {index}/{len(contestants)}: diversity for {name}')
    labels = load_labels(folders['labels'])
    reports[name]['final'] = build_final_report(reports[name], labels)
    failing_stems = [row['image'] for row in reports[name]['final']['failure']['images'] if row['failure']]
    reports[name]['final']['clip_visual_diversity'] = clip_visual_diversity(folders['generated'], failing_stems)
    save_report(reports[name], OUTPUT / f'{name}.json')
print('Diversity evaluation complete.')

Contestant 1/3: diversity for your_team_name
Contestant 2/3: diversity for your_team_name copy
Contestant 3/3: diversity for your_team_name copy 2
Diversity evaluation complete.


## 4. ⏱️ Efficiency

This section reports the translation runtime for each contestant using organizer-measured `generation_times.json`. The file must contain the standardized total generation time and may include the hardware used for measurement.

For each contestant, the evaluator records:

- **Total generation time:** the measured time for generating the submission.
- **Seconds per scene:** total generation time divided by the number of executed scenes.
- **Hardware:** the hardware metadata stored in `generation_times.json`.

If standardized timing is unavailable, efficiency is marked as unavailable and is not converted into an estimated value. These measurements are kept in each contestant report; the organizer-only notebook performs the cross-contestant Pareto ranking separately.

In [6]:
for index, (name, folders) in enumerate(contestants.items(), 1):
    final = reports[name]['final']
    executed = final['failure']['executed']
    reports[name]['efficiency'] = efficiency(folders['generated'].parent, executed)
    timing = reports[name]['efficiency'].get('seconds_per_scene')
    timing_text = f'{timing:.3f} s/scene' if timing is not None else 'unavailable'
    print(f'Contestant {index}/{len(contestants)}: {name} - {timing_text}')
    save_report(reports[name], OUTPUT / f'{name}.json')

print('Efficiency reports complete. Run the organizer-only Pareto ranking notebook to rank contestants.')

Contestant 1/3: your_team_name - unavailable
Contestant 2/3: your_team_name copy - unavailable
Contestant 3/3: your_team_name copy 2 - unavailable
Efficiency reports complete. Run the organizer-only Pareto ranking notebook to rank contestants.


## 5. 📄 Readable saved results

This cell reads the saved contestant reports and prints a compact summary. It can be run independently after the evaluation cells. The failure-rate label uses the selected `FAILURE_MODEL`; the organizer-only ranking notebook reads these saved reports to create the Pareto leaderboard.

In [7]:
for path in sorted(OUTPUT.glob('*.json')):
    if path.name == 'leaderboard.json':
        continue
    report = json.loads(path.read_text())
    print(f"\n{report['contestant']}")
    final = report.get('final')
    failure_model = report.get('failure_raw', {}).get('inference', {}).get('model', 'selected model')
    if final is None:
        raw = report.get('failure_raw', {})
        print(f"  Final report not assembled yet; {failure_model} scenes: {raw.get('executed', 0)}")
        continue
    failure = final['failure']
    failure_rate = failure.get('failure_rate')
    failure_text = f'{failure_rate:.3f}' if failure_rate is not None else 'N/A'
    print(f"  {failure_model} disagreement rate: {failure_text} ({failure['failures']}/{failure['executed']})")
    generated = final['realism']['generated_vs_real']['summary']
    simulated = final['realism']['simulated_vs_real']['summary']
    generated_ssim = generated.get('ssim')
    simulated_ssim = simulated.get('ssim')
    generated_text = f'{generated_ssim:.3f}' if generated_ssim is not None else 'N/A'
    simulated_text = f'{simulated_ssim:.3f}' if simulated_ssim is not None else 'N/A'
    print(f"  SSIM to real: generated {generated_text}, simulated {simulated_text}")
    sam = report.get('sam', {})
    sim_sam = sam.get('simulated_vs_generated', {}).get('summary', {}).get('partition', {})
    print(f"  SAM boundary F1: simulated_vs_generated {sim_sam.get('boundary_f1', float('nan')):.3f}")
    print(f"  Feature diversity: {final['feature_diversity']['mean_pairwise_hamming']:.3f}")
    timing = report.get('efficiency', {})
    seconds_per_scene = timing.get('seconds_per_scene')
    timing_text = f'{seconds_per_scene:.3f} s/scene' if seconds_per_scene is not None else 'N/A'
    print(f"  Generation time: {timing_text}")


provided_data_baseline
  Final report not assembled yet; vikhyatk/moondream2 scenes: 7

provided_data_baseline_no_translation
  Final report not assembled yet; selected model scenes: 0

your_team_name copy 2
  vikhyatk/moondream2 disagreement rate: N/A (0/0)
  SSIM to real: generated 0.377, simulated 0.377
  SAM boundary F1: simulated_vs_generated 1.000
  Feature diversity: 0.000
  Generation time: N/A

your_team_name copy
  vikhyatk/moondream2 disagreement rate: N/A (0/0)
  SSIM to real: generated 0.377, simulated 0.377
  SAM boundary F1: simulated_vs_generated 1.000
  Feature diversity: 0.000
  Generation time: N/A

your_team_name
  vikhyatk/moondream2 disagreement rate: N/A (0/0)
  SSIM to real: generated 0.377, simulated 0.377
  SAM boundary F1: simulated_vs_generated 1.000
  Feature diversity: 0.000
  Generation time: N/A
